In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
import math

In [12]:
# vocab:
# 0 = blank
# 1 = A
# 2 = B
# 3 = C
# 4 = D

logits = torch.tensor([
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=0 -> Blank
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # t=1 -> A
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=2 -> Blank
    [[0.0, 0.0, 10.0, 0.0, 0.0]],  # t=3 -> B
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=4 -> Blank
])

In [15]:
logits.shape

torch.Size([6, 1, 5])

### Greedy Search

In [7]:
def ctc_decode_greedy(logits, blank=0):
    predictions = logits.argmax(dim=-1)

    results = []

    for batch_idx in range(predictions.size(1)):
        tokens = predictions[:, batch_idx]

        decoded = []
        prev_token = None

        for token in tokens.tolist():
            if token == prev_token:
                continue
                
            if token != blank:
                decoded.append(token)

            prev_token = token

        results.append(decoded)
    return results    

In [8]:
ctc_decode_greedy(logits, blank=0)

[[1, 2]]

In [18]:
vocab = {
    0: "<blank>",
    1: "A",
    2: "B",
    3: "C",
    4: "D",
}

def index_to_vocab(decoded_indices, vocab):
    return [
        "".join(vocab[token] for token in sequence)
        for sequence in decoded_indices
    ]

In [20]:
index_to_vocab([[1, 2]], vocab)

['AB']

In [14]:
logits = torch.tensor([
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # -> Blank
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # -> A
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # -> A
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # -> Blank
    [[0.0, 0.0, 10.0, 0.0, 0.0]],  # -> B
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # -> Blank
])
ctc_decode_greedy(logits, blank=0)

[[1, 2]]

In [10]:
logits = torch.tensor([
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # -> Blank
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # -> A
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # -> Blank
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # -> A
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # -> Blank
    [[0.0, 0.0, 10.0, 0.0, 0.0]],  # -> B
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # -> Blank
])
ctc_decode_greedy(logits, blank=0)

[[1, 1, 2]]

In [11]:
logits = torch.tensor([
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # -> Blank
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # -> A
    [[0.0, 0.0, 10.0, 0.0, 0.0]],  # -> B
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # -> A
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # -> Blank
    [[0.0, 0.0, 10.0, 0.0, 0.0]],  # -> B
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # -> Blank
])
ctc_decode_greedy(logits, blank=0)

[[1, 2, 1, 2]]

### Beam Search